In [11]:
from importlib import import_module

named_libs = [('pandas', 'pd')] # (library_name, shorthand)
for (name, short) in named_libs:
    try:
        lib = import_module(name)
    except:
        print(sys.exc_info())
    else:
        globals()[short] = lib  

libnames = ['requests', 'json', 'pyexasol', 'configparser']
for libname in libnames:
    try:
        lib = import_module(libname)
    except:
        print(sys.exc_info())
    else:
        globals()[libname] = lib
        
### API EXTRACTION

def cleanNsafeExtract():
    try:
        ### Initialize the dataframe
        final_df = pd.DataFrame()
        ### API link
        url = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100"
        parsed = json.loads(requests.get(url).text)
        ### Get the number of pages through -> #int(parsed['total_pages'])
        for x in range(int(parsed['total_pages'])):
            url_iter = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page="+str(x+1)+"&size=100"
            response = requests.get(url_iter)
            if response.status_code == 403:    
                data = response.text
                parsed = json.loads(data)
                df = pd.DataFrame(parsed.get('results'))
                final_df = final_df.append(df, ignore_index=True)
            else: 
                print("Request failed page: {} ".format(x))
                
        final_df['created_date']= pd.to_datetime(final_df['created_date'])
        final_df['updated_date']= pd.to_datetime(final_df['updated_date'])
        harmonized_df = final_df[final_df.hkey.notnull()]
        #harmonized_df = harmonized_df.loc[:, harmonized_df.columns != 'checked']
    except:
        print('Failed in function cleanNsafe - ')
    return harmonized_df, print(f'Number of records having HOTEL_IDs {final_df.id[final_df.hkey.notnull()].count()}.'), print(f'Number of records with no HOTEL_IDs  {final_df.id[final_df.hkey.isna()].count()}.')    

In [12]:
def cleanNsafeLoad(df): 
    harmonized_df = df
    try:
        #Location of the ini file
        config = configparser.ConfigParser()
        ## Config location CHANGE
        config.read('C:\\Users\\svi02\\.spyder-py3\\pfxDET.ini')
        dsn=config['pfxDET']['dsn']
        user=config['pfxDET']['user']
        pwd=config['pfxDET']['pwd']
        schema=config['pfxDET']['schema']
        # Exasol connection
        connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
        connect.execute("TRUNCATE TABLE DWHPFX.CLEAN_SAFE_HOTELS")
        connect.import_from_pandas(harmonized_df, table = ('DWHPFX','CLEAN_SAFE_HOTELS'))
        stmt = connect.last_statement()
        print(f'Number of records inserted  {stmt.rowcount()}.')
    except:
        print('Failed in function cleanNsafeLoad - check DB connection')
#     return print(f'Number of records inserted  {harmonized_df.id[harmonized_df.hkey.notnull()].count()}.')



In [13]:
df, a, b = cleanNsafeExtract()
cleanNsafeLoad(df)
df.head()

Number of records having HOTEL_IDs 57597.
Number of records with no HOTEL_IDs  59.
Failed in function cleanNsafeLoad - check DB connection


,id,hkey,name,created_date,updated_date,checked,link,program_name,version,terms,...,brand_id,brand_name,city_id,city,country_id,country,performance_cluster,status,type,missed
0,00005b15-b99c-444d-97bf-d3cd02f8c8db,431251.0,Homewood Suites by Hilton Atlanta I-85-Lawrenc...,2020-07-31 16:16:24+00:00,2020-07-31 16:16:24+00:00,"1,2,3,4,5,6,8,10,11,12,13,14,15,16,17,18,19,20...",https://www.hilton.com/en/corporate/cleanstay/,Hilton CleanStayTM,1,2,...,1537.0,Homewood Suites by Hilton,168172.0,Lawrenceville (Georgia),165.0,USA,4.0,True,cleansafe_self_inspection,
1,00016f7b-171e-4be7-900a-5d0f10bd9018,121783.0,Clarion Hotel Kansas City - Overland Park,2020-08-13 18:49:23+00:00,2020-08-13 18:49:23+00:00,"1,2,4,5,6,8,10,11,12,14,15,18,19,27,28,32,33,3...",https://www.radissonhotels.com/en-us/social-re...,SGS - 20 & 10 steps Protocols,1,2,...,1094.0,Radisson Hotel Group (Opt-in 14%),163963.0,Lenexa (Kansas),165.0,USA,NaN,True,cleansafe_expert_inspection,
2,0001db0c-cfb3-4c83-b841-67bbf7f002f0,404054.0,Quality Suites Stratford,2020-09-03 19:39:02+00:00,2020-09-03 19:39:02+00:00,None,https://www.hotel-audit.hrs.com/clean-and-safe,Clean & Safe Protocol,1,2,...,1760.0,Quality by Choice,141060.0,Stratford (Connecticut),165.0,USA,5.0,False,cleansafe_self_inspection,"1,2,4,5,6,8,10,11,12,14,15,18,19,27,28,39,40,4..."
3,000225aa-e50c-46c0-a3fe-65813215f837,442825.0,Hampton Inn - Suites Mount Pleasant,2020-07-31 16:16:24+00:00,2020-07-31 16:16:24+00:00,"1,2,3,4,5,6,8,10,11,12,13,14,15,16,17,18,19,20...",https://www.hilton.com/en/corporate/cleanstay/,Hilton CleanStayTM,1,2,...,1535.0,Hampton Inn by Hilton,190895.0,Mount Pleasant (Texas),165.0,USA,5.0,True,cleansafe_self_inspection,
4,000311b7-76c8-46f3-a9d5-9a82cc5b813d,863579.0,ibis Styles Barcelona City Bogatell,2020-08-28 08:06:14+00:00,2020-08-28 08:06:14+00:00,"1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,2...",https://www.all.accor.com/event/allsafe.en.shtml,ALLSAFE,1,2,...,1602.0,IBIS Styles Standard,64430.0,Barcelona (Catalunya),140.0,Spain,2.0,True,cleansafe_expert_inspection,


In [79]:
from urllib.parse import urlsplit
parsed = urlsplit("https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100")
print('query  :', parsed.query)

query  : secretKey=3AwuZKz9nH&page=1&size=100


In [80]:
print('query  :', parsed.fragment)

query  : 


In [6]:
### Initialize the dataframe
final_df = pd.DataFrame()
### API link
url = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100"
parsed = json.loads(requests.get(url).text)
### Get the number of pages through -> #int(parsed['total_pages'])
for x in range(int(parsed['total_pages'])):
    url_iter = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page="+str(x+1)+"&size=100"
    response = requests.get(url_iter)
    if response.status_code == 403:    
        data = response.text
        parsed = json.loads(data)
        df = pd.DataFrame(parsed.get('results'))
        final_df = final_df.append(df, ignore_index=True)
    else: 
        print("Request failed page: {} ".format(x))

final_df['created_date']= pd.to_datetime(final_df['created_date'])
final_df['updated_date']= pd.to_datetime(final_df['updated_date'])
harmonized_df = final_df[final_df.hkey.notnull()]

In [7]:
harmonized_df.count()

id                     57088
hkey                   57088
name                   57041
created_date           57088
updated_date           57087
checked                38313
link                   57088
program_name           57088
version                57088
terms                  57088
services               31485
audit_date              5336
auditor_key             3310
chain_id               57041
chain_name             57029
brand_id               57041
brand_name             57029
city_id                57041
city                   57041
country_id             57041
country                57041
performance_cluster    36111
status                 57088
type                   57088
missed                 57088
dtype: int64

In [8]:
cleanNsafeLoad(df)

Failed in function cleanNsafeLoad - check DB connection


In [14]:
config = configparser.ConfigParser()
## Config location CHANGE
config.read('C:\\Users\\svi02\\.spyder-py3\\pfxDET.ini')
dsn=config['pfxDET']['dsn']
user=config['pfxDET']['user']
pwd=config['pfxDET']['pwd']
schema=config['pfxDET']['schema']
# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
# df = connect.export_to_pandas("SELECT * FROM DWHBIL.V_LKP_HOTEL LIMIT 100")
# df.head()
connect.import_from_pandas(df, table = ('DWHPFX','CLEAN_SAFE_HOTELS'))
stmt = connect.last_statement()

BrokenPipeError: [Errno 32] Broken pipe